# 10 – Check: does the model approve loans people can't afford?
Notebooks 03–07 looked for people who are **wrongly rejected**. This one looks the other way: people
who are **wrongly approved**.

We have no repayment data, so we use a standard affordability measure. **EMI as a share of monthly
income** (lenders call it FOIR) is the monthly instalment divided by monthly income. An EMI of 100%
of income means the whole salary goes to this one loan, before rent, food, or the applicant's other
loans.

The model never sees the interest rate, so we **assume 14% a year** to work out the EMI. The exact
rate barely matters here: at any realistic rate, an EMI equal to the whole income is unaffordable.

In [1]:
%matplotlib inline
import pandas as pd
from common import load_data, predict, what_if

RATE = 0.14 / 12   # assumed monthly interest rate


def emi(loan, months):
    return loan * RATE * (1 + RATE) ** months / ((1 + RATE) ** months - 1)


def loan_for_emi_share(share, income, months):
    """The loan amount whose EMI equals `share` of monthly income."""
    return (share * income) * ((1 + RATE) ** months - 1) / (RATE * (1 + RATE) ** months)


df = load_data()
df["emi_share"] = emi(df.loan_amount, df.tenure_months) / df.monthly_income
df["label_rejected"] = df.decision == "reject"

## 1. How did the old decisions treat unaffordable loans?

In [2]:
df.groupby(pd.cut(df.emi_share, [0, 0.5, 0.6, 0.7, 0.8, 1.0, 2.0])).label_rejected.agg(["count", "mean"]).round(3)

,count,mean
emi_share,,
"(0.0, 0.5]",453,0.486
"(0.5, 0.6]",41,0.902
"(0.6, 0.7]",29,0.828
"(0.7, 0.8]",25,0.920
"(0.8, 1.0]",36,1.000
"(1.0, 2.0]",16,1.000


The old credit team rejected **every** applicant whose EMI was over 80% of income (52 of 52).
So the training data did contain a clear affordability rule.

## 2. Does the model follow it?
Take the applicants the model approves. Change **only** the loan amount, so that the EMI becomes a
given share of their income, and keep everything else the same.

In [3]:
_, rejected = predict(df)
approved = df[~rejected]
print("Model-approved applicants:", len(approved))

rows = []
for share in [0.3, 0.5, 0.6, 0.8, 1.0, 1.5]:
    loan = loan_for_emi_share(share, approved.monthly_income, approved.tenure_months).round(-3)
    in_range = loan <= 4000000          # stay within the loan sizes seen in the data
    _, r = what_if(approved[in_range], loan_amount=loan[in_range])
    rows.append({"EMI as % of income": f"{share:.0%}", "applicants": in_range.sum(),
                 "still_approved": round(1 - r.mean(), 3)})
pd.DataFrame(rows)

Model-approved applicants: 245


,EMI as % of income,applicants,still_approved
0,30%,245,0.922
1,50%,243,0.691
2,60%,240,0.529
3,80%,232,0.198
4,100%,218,0.078
5,150%,185,0.011


**What to notice:** the old decisions rejected everyone above 80%. The model still approves about
**1 in 5 at 80%** and about **1 in 13 when the EMI is their entire income**.

## 3. Who gets approved with an EMI equal to their whole income?

In [4]:
loan = loan_for_emi_share(1.0, approved.monthly_income, approved.tenure_months).round(-3)
test = approved.assign(loan_amount=loan)
test = test[test.loan_amount <= 4000000]
_, r = predict(test)
cols = ["age", "monthly_income", "loan_amount", "tenure_months", "employment_years",
        "credit_score", "existing_loans"]
print(f"{(~r).sum()} applicants approved with EMI = 100% of income")
test[~r][cols].describe().loc[["min", "50%", "max"]].round(0)

17 applicants approved with EMI = 100% of income


,age,monthly_income,loan_amount,tenure_months,employment_years,credit_score,existing_loans
min,29.0,14438.0,441000.0,12.0,0.0,747.0,1.0
50%,45.0,28904.0,990000.0,48.0,8.0,818.0,2.0
max,62.0,98374.0,1943000.0,60.0,14.0,885.0,3.0


They are mostly **lower-income** applicants (median about ₹29,000 a month), the people least able to absorb an unaffordable loan, and all of them have 1–3 other active loans. All of them have **high credit scores (747–885)**: the model lets a good credit score override affordability.

## 4. Other checks for over-lenient approvals
Same approved applicants, one weak signal at a time.

In [5]:
checks = {
    "credit score 550": {"credit_score": 550},
    "credit score 600": {"credit_score": 600},
    "6 existing loans": {"existing_loans": 6},
    "0 years in job": {"employment_years": 0},
    "income Rs10,000 (loan unchanged)": {"monthly_income": 10000},
}
rows = []
for name, change in checks.items():
    _, r = what_if(approved, **change)
    rows.append({"change": name, "still_approved": round(1 - r.mean(), 3)})
pd.DataFrame(rows)

,change,still_approved
0,credit score 550,0.000
1,credit score 600,0.065
2,6 existing loans,0.678
3,0 years in job,0.853
4,"income Rs10,000 (loan unchanged)",0.380


A low credit score is handled sensibly (almost no approvals at 550–600). Cutting income to ₹10,000
while keeping the same loan still leaves many approved. That is the same affordability gap seen from
the income side.

**Conclusion:** affordability is **mostly handled correctly**. Approvals fall steadily as the EMI rises, and no real approved applicant has an EMI above 80% of income. One weak spot: at extreme EMIs (80–100% of income), a high credit score can still get a loan approved, while the old credit team rejected all such cases. This is recorded in FINDINGS.md under "Things I tested that turned out fine", not as a finding.